# JAX Swarm ES: TPU run

Runs the full TPU study and pushes the results back to the repo:

1. **Throughput:** drone and pendulum env-steps/s from 1 to 262,144 parallel envs, compared with the committed CPU and T4 numbers.
2. **Drone:** trains the waypoint-chasing drone (`--pop 2048 --n-eval 5 --gens 500`) and renders episodes.
3. **Pendulum seed studies:** the same config trained from 12 seeds, for three setups (small population, large population, curriculum). It reports how many seeds actually solve the swing-up, because ES on the pendulum is bimodal: some seeds solve it and others get stuck.

**Before you run**
- *Runtime → Change runtime type →* **v5e-1 TPU** → Save.
- *Key icon (Secrets)* → add `GITHUB_TOKEN` (a fine-grained token for `shiroonigami23-ui/jax-swarm-es` with *Contents: read and write*) and switch on **Notebook access**. Without it the results stay in the Colab session and are not pushed.
- Then use *Runtime → Run all*. It takes about 30–60 minutes. Keep the tab open.

In [ ]:
# Check that the TPU is visible
import subprocess, sys
print(subprocess.run([sys.executable, "-c", "import jax; print(jax.devices())"], capture_output=True, text=True).stdout or "jax not installed yet, the runner installs it")

In [ ]:
# Get the code (or update it if already cloned)
import os
if not os.path.exists("/content/jax-swarm-es/.git"):
    !git clone -q https://github.com/shiroonigami23-ui/jax-swarm-es /content/jax-swarm-es
else:
    !git -C /content/jax-swarm-es pull -q --ff-only
%cd /content/jax-swarm-es

In [ ]:
# Full TPU study: install -> tests -> benchmarks -> drone -> pendulum seed studies -> renders -> README -> push
%run /content/jax-swarm-es/colab_run.py --tpu

## Look at the results here

The runner has already updated the README section `TPU-RESULTS` and pushed everything under `results/tpu/`. The next cell shows the main numbers in the notebook.

In [ ]:
import json
from IPython.display import Image, display, Markdown
s = json.load(open("results/tpu/summary.json"))
print("device:", s["device"])
for name, st in s["studies"].items():
    print(f"{name:11s} solved {round(st['success_rate']*st['n_seeds'])}/{st['n_seeds']}  median {st['final_eval_median']:.1f}  best {st['final_eval_best']:.1f}")
display(Image("results/tpu/throughput_all.png"))
for name in s["studies"]:
    display(Markdown(f"**{name}**")); display(Image(f"results/tpu/seeds_{name}/curves.png"))